# 03. 작은 Isaac Sim — ROS 2 × PyBullet 브리지 (Google Colab)

이 노트북은 앞의 두 노트북을 하나로 합칩니다. Isaac Sim 의 핵심 아이디어 —

```
정책(policy) ──토픽──▶ 시뮬레이터 ──토픽──▶ 정책
```

를 **Colab 한 방** 에서 체험합니다. GPU 불필요.

### 파이프라인

```
┌─────────────────────┐      /joint_cmd       ┌──────────────────────┐
│ 클라이언트 노드      │ ───사인파 궤적────▶  │  PyBullet 브리지      │
│ (사인파 커맨드)      │                       │  Panda 시뮬레이션     │
└─────────────────────┘      /joint_states    │  카메라 캡처           │
            ▲ ──────────────실측 관절 각────  └──────────────────────┘
```

결과물: (1) 관절 궤적 플롯, (2) 명령 vs 실측 추적 오차 플롯, (3) 시뮬 카메라 애니메이션.

## 1. 설치

런타임당 1회. ROS 2 Humble 을 이미 설치한 셀이 없다면 첫 번째 셀부터 실행.

In [ ]:
%%bash
if ! command -v ros2 &> /dev/null; then
  sudo apt update -qq && sudo apt install -y -qq software-properties-common curl gnupg locales > /dev/null
  sudo locale-gen en_US.UTF-8 > /dev/null
  sudo add-apt-repository -y universe > /dev/null 2>&1
  sudo curl -sSL https://raw.githubusercontent.com/ros/rosdistro/master/ros.key -o /usr/share/keyrings/ros-archive-keyring.gpg
  echo "deb [arch=$(dpkg --print-architecture) signed-by=/usr/share/keyrings/ros-archive-keyring.gpg] http://packages.ros.org/ros2/ubuntu $(. /etc/os-release && echo $UBUNTU_CODENAME) main" | sudo tee /etc/apt/sources.list.d/ros2.list > /dev/null
  sudo apt update -qq && sudo apt install -y -qq ros-humble-ros-base ros-humble-sensor-msgs > /dev/null
  echo 'ROS 2 Humble 새로 설치'
else
  echo 'ROS 2 이미 설치됨'
fi
pip -q install pybullet==3.2.7 imageio[ffmpeg] numpy matplotlib pandas pillow

## 2. 브리지 노드 작성 — PyBullet + ROS 2

`/joint_cmd` 를 받아 PyBullet 에 넣고, `/joint_states` 로 실측값을 발행. 프레임은 `frames/` 폴더에 저장해서 나중에 애니메이션으로 합칩니다.

In [ ]:
%%writefile sim_bridge.py
import os, numpy as np, rclpy
from rclpy.node import Node
from sensor_msgs.msg import JointState
import pybullet as p, pybullet_data, imageio.v2 as imageio


class PandaSimBridge(Node):
    N_ARM = 7

    def __init__(self):
        super().__init__('panda_sim_bridge')
        os.makedirs('frames', exist_ok=True)
        p.connect(p.DIRECT)
        p.setAdditionalSearchPath(pybullet_data.getDataPath())
        p.setGravity(0, 0, -9.81)
        p.loadURDF('plane.urdf')
        self.robot = p.loadURDF('franka_panda/panda.urdf', useFixedBase=True)

        self.pub = self.create_publisher(JointState, '/joint_states', 10)
        self.create_subscription(JointState, '/joint_cmd', self.cb, 10)
        self.create_timer(1/240, self.step)
        self.frame_i = 0

    def cb(self, msg: JointState):
        for i, q in enumerate(msg.position[:self.N_ARM]):
            p.setJointMotorControl2(self.robot, i, p.POSITION_CONTROL,
                                     targetPosition=q, force=87)

    def step(self):
        p.stepSimulation()
        js = JointState()
        js.header.stamp = self.get_clock().now().to_msg()
        js.name = [f'panda_joint{i+1}' for i in range(self.N_ARM)]
        js.position = [p.getJointState(self.robot, i)[0] for i in range(self.N_ARM)]
        self.pub.publish(js)

        # 10 step 마다 (24 FPS) 카메라 프레임 저장
        if self.frame_i % 10 == 0:
            view = p.computeViewMatrix([1.2, 1.2, 1.0], [0, 0, 0.3], [0, 0, 1])
            proj = p.computeProjectionMatrixFOV(60, 4/3, 0.05, 10)
            _, _, rgba, _, _ = p.getCameraImage(320, 240, view, proj,
                                                 renderer=p.ER_BULLET_HARDWARE_OPENGL)
            rgb = np.reshape(rgba, (240, 320, 4))[:, :, :3].astype('uint8')
            imageio.imwrite(f'frames/{self.frame_i:05d}.png', rgb)
        self.frame_i += 1


def main():
    rclpy.init()
    rclpy.spin(PandaSimBridge())


if __name__ == '__main__':
    main()


## 3. 클라이언트 — 사인파 궤적을 명령으로 발행

In [ ]:
%%writefile client.py
import math, csv, time, rclpy
from rclpy.node import Node
from sensor_msgs.msg import JointState


class Commander(Node):
    def __init__(self):
        super().__init__('commander')
        self.pub = self.create_publisher(JointState, '/joint_cmd', 10)
        self.create_subscription(JointState, '/joint_states', self.on_state, 10)
        self.create_timer(1/50, self.tick)   # 50 Hz 명령
        self.t0 = time.time()
        self.log = open('bridge_log.csv', 'w', buffering=1)
        self.w = csv.writer(self.log)
        self.w.writerow(['t', 'kind'] + [f'q{i+1}' for i in range(7)])

    def tick(self):
        t = time.time() - self.t0
        q = [0.7*math.sin(1.2*t),
             -0.5+0.3*math.sin(1.0*t),
             0.0,
             -2.0+0.4*math.sin(0.8*t),
             0.0,
             1.2+0.3*math.sin(1.5*t),
             0.5*math.sin(1.0*t)]
        msg = JointState()
        msg.header.stamp = self.get_clock().now().to_msg()
        msg.name = [f'panda_joint{i+1}' for i in range(7)]
        msg.position = q
        self.pub.publish(msg)
        self.w.writerow([f'{t:.3f}', 'cmd'] + [f'{x:.4f}' for x in q])

    def on_state(self, msg: JointState):
        t = time.time() - self.t0
        self.w.writerow([f'{t:.3f}', 'act'] + [f'{x:.4f}' for x in msg.position[:7]])


def main():
    rclpy.init()
    node = Commander()
    try:
        rclpy.spin(node)
    finally:
        node.log.close()


if __name__ == '__main__':
    main()


## 4. 두 노드 실행 — 6초 간 기록

In [ ]:
import subprocess, os, time, signal, shutil

shutil.rmtree('frames', ignore_errors=True)

os.environ['ROS_DOMAIN_ID'] = '7'

def spawn(cmd):
    return subprocess.Popen(
        ['bash', '-lc', 'source /opt/ros/humble/setup.bash && ' + cmd],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        preexec_fn=os.setsid,
    )

bridge = spawn('python sim_bridge.py')
time.sleep(1.5)     # bridge 가 토픽 광고 먼저
client = spawn('python client.py')

try:
    time.sleep(6.0)
finally:
    for proc in (client, bridge):
        os.killpg(os.getpgid(proc.pid), signal.SIGINT)
    time.sleep(1.0)
print(f'frames 캡처: {len(os.listdir("frames"))}장, 로그 저장 완료')

## 5. 명령 vs 실측 — 추적 성능 시각화

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

df = pd.read_csv('bridge_log.csv')
cmd = df[df.kind=='cmd'].reset_index(drop=True)
act = df[df.kind=='act'].reset_index(drop=True)
print(f'cmd rows = {len(cmd)}, act rows = {len(act)}')

fig, axes = plt.subplots(4, 2, figsize=(11, 9), sharex=True)
axes = axes.ravel()
for i in range(7):
    ax = axes[i]
    ax.plot(cmd['t'], cmd[f'q{i+1}'], label='cmd', lw=1)
    ax.plot(act['t'], act[f'q{i+1}'], label='act', lw=1, alpha=0.8)
    ax.set_title(f'panda_joint{i+1}')
    ax.grid(alpha=0.3)
    if i == 0:
        ax.legend()
axes[-1].axis('off')
fig.suptitle('/joint_cmd vs /joint_states — 추적 성능', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# 관절별 평균 절대 추적 오차
import numpy as np

cmd_i = cmd.set_index('t').iloc[:, 1:]
act_i = act.set_index('t').iloc[:, 1:]
merged = cmd_i.join(act_i, lsuffix='_cmd', rsuffix='_act').dropna()

errs = {f'q{i+1}': (merged[f'q{i+1}_cmd'] - merged[f'q{i+1}_act']).abs().mean()
        for i in range(7)}
pd.Series(errs, name='mean_abs_err_rad').to_frame().plot.bar(figsize=(7, 3), legend=False)
plt.title('관절별 평균 절대 추적 오차 [rad]')
plt.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 6. 시뮬 카메라 프레임 → GIF

In [ ]:
import glob, imageio.v2 as imageio
from IPython.display import Image, display

files = sorted(glob.glob('frames/*.png'))
imgs = [imageio.imread(f) for f in files[::2]]   # 2프레임 건너뛰기
gif = 'bridge.gif'
imageio.mimsave(gif, imgs, fps=24, loop=0)
with open(gif, 'rb') as f:
    display(Image(data=f.read(), format='gif'))

## 7. 여기서 무엇을 배웠나

- **분리된 프로세스** — 정책/클라이언트와 시뮬레이터가 서로를 몰라도 **토픽 이름** 만 맞으면 연결됩니다. Isaac Sim + ROS 브리지 구성과 완전히 같은 구조.
- **토픽 = 인터페이스** — `JointState` 하나로 명령/실측이 양쪽 방향 모두 흐릅니다.
- **시각화 3종 세트**:
  1. cmd vs act 겹쳐 그리기 → 제어 품질 파악
  2. 관절별 평균 오차 bar → 어느 관절이 힘든지
  3. 카메라 GIF → 로봇 움직임을 눈으로 확인

### 다음은
- Panda 대신 **UR5**, **KUKA IIWA** URDF 로 교체 → PyBullet 데이터 경로에 모두 들어있음
- `JointState` 대신 **`geometry_msgs/Twist`** 로 모바일 로봇 제어
- Colab GPU 인스턴스에서 **Stable-Baselines3** + gymnasium PyBullet 환경으로 RL 학습
- 진짜 Isaac Sim 이 필요할 땐 NVIDIA Brev Launchable 로 클라우드 접속 (튜토리얼 본편 § 7)